Validation task reads persisted target table. It does not depend on notebook local variables from earlier tasks

In [ ]:
from pyspark.sql import functions as F

silver_table =  "workspace.incremental_project.orders_silver_step4"

silver_df = spark.table(silver_table)

F.col("order_amount") < 0 tests a business rule that order amounts must not be negative

In [ ]:
total_rows = silver_df.count()

unique_orders = (
    silver_df.select("order_id")
    .distinct()
    .count()
)

null_ids = silver_df.filter(
    F.col("order_id").isNull()
).count()

invalid_amounts = silver_df.filter(
    F.col("order_amount").isNull()
    | (F.col("order_amount") < 0)
).count()

invalid_timestamps = silver_df.filter(
    F.col("updated_at").isNull()
).count()

print("Total:", total_rows)
print("Unique:", unique_orders)
print("Null IDs:", null_ids)
print("Invalid amounts:", invalid_amounts)
print("Invalid timestamps:", invalid_timestamps)

If an assertion fails, notebook task fails

In [ ]:
assert total_rows >= 8, "Unexpected row count"
assert total_rows == unique_orders, "Duplicate order IDs"
assert null_ids == 0, "Missing order IDs"
assert invalid_amounts == 0, "Invalid amounts"
assert invalid_timestamps == 0, "Invalid timestamps"

print("ALL DATA QUALITY CHECKS PASSED")

In [ ]:
bronze = spark.table(
    "workspace.incremental_project.orders_bronze_v3"
)

silver = spark.table(
    "workspace.incremental_project.orders_silver_step4"
)

print("Bronze rows:", bronze.count())
print("Silver rows:", silver.count())

assert bronze.count() == 10
assert silver.count() == 8

print("Workflow validation passed")

Orchestration coordinates multiple data processing tasks in correct sequence. We implemented it in Step 7 using Lakeflow Jobs and task dependencies between Bronze, Silver, and Quality

We use task dependencies to enforce execution order and prevent downstream tasks from running before their upstream processing succeeds

AvailableNow processes the available input data in a finite streaming run. We use awaitTermination() to wait for run to finish, while Lakeflow task dependencies coordinate downstream execution

When silver fails, dependent Quality task is skipped under the All succeeded condition. This prevents downstream validation from running against potentially stale data

Declarative Automation Bundles let us define Databricks jobs and related resources as code, enabling version control, automated validation, and repeatable deployments across environments